# 2D histograms

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

`Histogram2D` bins scatter `(x, y)` data into a 2D density grid rendered as a color map. Useful for phase-space distributions (Vx vs Vy), correlation plots (Bx vs By), or any scatter where you want to see density rather than individual points.

**What you'll learn**
- Create a 2D histogram from static numpy arrays.
- Use a log color scale.
- Update the data on the fly (`set_data`) and overlay several histograms on the same axes.
- Drive a histogram with a callback so SciQLop re-bins on pan/zoom.

**Prerequisites** — [Plot panels](5-SciQLopBasicPlotPanel.ipynb), [Speasy first steps](../Speasy/2-SpeasyFirstSteps.ipynb).

**Next** — [Annotation layers](16-SciQLopAnnotationLayers.ipynb).


## 1. Creating a histogram from scatter data

In [ ]:
import numpy as np
from SciQLop.user_api.plot import create_plot_panel

# Generate sample scatter data: a noisy ring
rng = np.random.default_rng(42)
theta = rng.uniform(0, 2 * np.pi, 50_000)
r = 1.0 + 0.3 * rng.standard_normal(50_000)
x = r * np.cos(theta)
y = r * np.sin(theta)

p = create_plot_panel()
xy_plot, hist = p.histogram2d(x, y, x_bins=80, y_bins=80)


## 2. Logarithmic color scale

When counts span several orders of magnitude, a log scale improves contrast.

In [ ]:
hist.z_log_scale = True

## 2b. Changing the colour gradient

The `.gradient` setter accepts a `SciQLopPlots.ColorGradient` value. Available presets include `Grayscale`, `Hot`, `Cold`, `Night`, `Candy`, `Geography`, `Ion`, `Thermal`, `Polar`, `Spectrum`, `Jet`, `Hues`.


In [ ]:
from SciQLopPlots import ColorGradient

hist.gradient = ColorGradient.Thermal


## 3. Updating data

Call `set_data(x, y)` to re-bin with new scatter data without recreating the plot.

In [ ]:
# Two gaussian clusters
x2 = np.concatenate([rng.normal(-1, 0.5, 20_000), rng.normal(1, 0.3, 20_000)])
y2 = np.concatenate([rng.normal(0, 0.5, 20_000), rng.normal(1, 0.3, 20_000)])
hist.set_data(x2, y2)

## 4. Visibility

In [ ]:
hist.visible = False
hist.visible = True

## 5. Callable histogram (live update on pan/zoom)

Instead of passing static arrays, pass a **callback** function. SciQLop calls it whenever the time range changes — `(start, stop)` are epoch floats, same as virtual products — and you return the `(x, y)` arrays to bin.

A practical detail: if you bin raw signal values, the histogram's axis range drifts with the data every time you pan. Normalising inside the callback (here we divide Bx, By by |B|) pins the axes to `[-1, 1]` and turns the density plot into a *direction* distribution — much steadier to read while panning. The companion time-series subplot below carries the magnitude information that this normalisation throws away.

> **Empty-data safety** — return empty arrays (`np.array([]), np.array([])`) when there is nothing to plot in the current interval. Histogram2D handles that case gracefully, and the same defensive style is a good habit for any callback-driven graph.


In [ ]:
import numpy as np
import speasy as spz
from SciQLop.user_api.plot import create_plot_panel
from SciQLop.user_api import TimeRange


def bxy_direction(start, stop):
    """Return (Bx/|B|, By/|B|) — the magnetic field direction in the GSE X-Y plane.

    Normalising fixes the histogram axes in [-1, 1] regardless of |B|, so the
    density plot doesn't shift as you pan. MMS FGM survey L2 carries |B| as the
    4th column, so we read it directly instead of recomputing it.
    """
    b = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b is None:
        return np.array([]), np.array([])
    bx, by, bmag = b.values[:, 0], b.values[:, 1], b.values[:, 3]
    safe = bmag > 0
    return (bx[safe] / bmag[safe]).astype(np.float64), (by[safe] / bmag[safe]).astype(np.float64)


p3 = create_plot_panel()
p3.time_range = TimeRange("2017-11-20T02:30:00", "2017-11-20T04:00:00")

# 1. Bx/|B| vs By/|B| direction density — re-binned on pan/zoom, axes pinned to [-1, 1].
_, live_hist = p3.histogram2d(bxy_direction, x_bins=80, y_bins=80, z_log_scale=True)

# 2. The raw B-field as a time series underneath — carries magnitude info that
#    the normalised histogram throws away.
p3.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")


## 6. Real-data example: Bx vs By correlation

Plot the correlation between two magnetic field components using static data.


In [ ]:
import speasy as spz

b_gse = spz.get_data(
    "cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2",
    "2017-11-20T02:30:00", "2017-11-20T04:00:00",
)

if b_gse is not None:
    bx = b_gse.values[:, 0]
    by = b_gse.values[:, 1]

    p4 = create_plot_panel()
    _, corr_hist = p4.histogram2d(bx, by, x_bins=100, y_bins=100, z_log_scale=True)


## API reference

| Method / Property | Description |
|---|---|
| `panel.histogram2d(x, y, *, x_bins, y_bins, z_log_scale)` | Create a histogram from static data in a new plot |
| `panel.histogram2d(callback, *, x_bins, y_bins, z_log_scale)` | Create a histogram with a live callback in a new plot |
| `hist.set_data(x, y)` | Update scatter data (static histograms only) |
| `.z_log_scale` | Toggle log color scale |
| `.visible` | Show/hide the histogram |
| `.gradient` | Color gradient (`SciQLopPlots.ColorGradient`) |

**One colormap per plot.** A plot has a single color-scale axis. Each call to `panel.histogram2d(...)` adds the histogram in its own new plot. Calling `xy_plot.histogram2d(...)` or `time_series_plot.histogram2d(...)` on a plot that already carries a colormap-style plottable (Histogram2D, ColorMap, Waterfall) raises a `RuntimeError`.

**Callable signature:** `f(start, stop) -> (x, y)` where `start` and `stop` are epoch floats (same convention as virtual products).
